# 03 – Person Details

Esplorazione e data cleaning del dataset `person_details.csv`.

**Colonne:**

|---|---|
| Colonna | Descrizione |
| `person_mal_id` | ID univoco della persona su MyAnimeList|
| `url` | URL della pagina MAL della persona |
| `website_url` | Sito web personale |
| `image_url` | URL dell'immagine del profilo |
| `name` | Nome completo |
| `given_name` | Nome proprio |
| `family_name` | Cognome |
| `birthday` | Data di nascita |
| `favorites` | Numero di utenti che l'hanno tra i preferiti |
| `relevant_location` | Posizione |

## 1. Import e caricamento dati
Importiamo le librerie necessarie e carichiamo il file csv. Facciamo una esplorazione generica per capire la struttura e le caratteristiche del dataset.

In [ ]:
import pandas as pd
import numpy as np
from dataset_analyzer import analyze
df_pd = pd.read_csv('../datasets/person_details.csv')
print(f'Shape: {df_pd.shape}')
print()
df_pd.info()
df_pd.head()

## 1.1 Rimozione duplicati esatti

Prima dell'analisi per colonna, rimuoviamo le righe con valori identici in **tutte** le colonne, mantenendo solo la prima occorrenza.

In [1]:
n_originale = len(df_pd)

mask_dup = df_pd.duplicated(keep=False)
n_righe_coinvolte = mask_dup.sum()
n_gruppi = df_pd[mask_dup].duplicated(keep='first').sum()
n_tenute = n_righe_coinvolte - n_gruppi

print(f'Righe totali coinvolte in duplicazioni : {n_righe_coinvolte:,}')
print(f'  → prime occorrenze mantenute         : {n_tenute:,}')
print(f'  → occorrenze extra rimosse           : {n_gruppi:,}')
print()

df_pd.drop_duplicates(keep='first', inplace=True)
print(f'Righe prima della rimozione : {n_originale:,}')
print(f'Righe dopo la rimozione     : {len(df_pd):,}')

NameError: name 'df_pd' is not defined

## 2. Analisi colonna per colonna

### 2.1 `person_mal_id`

ID univoco della persona su MAL. È la **chiave primaria** del dataset. I valori duplicati **non sono attesi**.

In [ ]:
analyze(df_pd['person_mal_id'])

**Osservazioni:**
- Nessun null
- Il dtype è già `int64` quindi nessuna conversione necessaria.
- Si nota che 99.87% dei valori sono univoci. Stampiamo un campione per verificare i duplicati.

In [ ]:
print(f'Null in person_mal_id      : {df_pd["person_mal_id"].isna().sum()}')
print(f'Duplicati in person_mal_id : {df_pd["person_mal_id"].duplicated().sum()}')

# Campione righe duplicate
mask_pk_dup = df_pd.duplicated(subset=['person_mal_id'], keep=False)
df_dup = df_pd[mask_pk_dup].sort_values('person_mal_id')
print(f'\nRighe coinvolte: {len(df_dup)} | person_mal_id unici duplicati: {df_dup["person_mal_id"].nunique()}')
df_dup.head(20)

I duplicati consistono in righe in cui l'unico valore che cambia è `relevant_location`. Non possiamo rimuovere le righe in quanto non è possibile sapere quale location è quella corretta. In più, la presenza di questa colonna rompe la struttura della tabella dove ogni persona dovrebbe comparire in una sola riga dove la chiave primaria è   `person_mal_id`. Abbiamo deciso di rimuovere la colonna `relevant_location` e infine rimuovere le righe duplicate mantenendo solo la prima occorrenza.

In [ ]:
# Rimozione colonna
df_pd.drop(columns=['relevant_location'], inplace=True)

# Rimozione duplicati indotti (ora sono duplicati esatti)
n_prima = len(df_pd)
df_pd.drop_duplicates(subset=['person_mal_id'], keep='first', inplace=True)
df_pd.reset_index(drop=True, inplace=True)
print(f'Righe prima : {n_prima:,}')
print(f'Righe dopo  : {len(df_pd):,}')
print(f'Rimosse     : {n_prima - len(df_pd):,}')